# QLoRA Summarization — Colab Training Pipeline

In [ ]:
GITHUB_REPO = "https://github.com/ducnd58233/language-engineer.git"
GITHUB_TOKEN = ""
HF_TOKEN = ""
HF_REPO_ID = "ducnd58233/qwen2.5-3b-qlora-summarization"
EPOCHS = 1
N_EVAL_SAMPLES = 100

## 1. Setup

In [ ]:
import os
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or GITHUB_TOKEN
except Exception:
    pass

repo_url = GITHUB_REPO.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else GITHUB_REPO

if not os.path.exists("language-engineer"):
    subprocess.run(["git", "clone", repo_url, "language-engineer"], check=True)
else:
    subprocess.run(["git", "-C", "language-engineer", "pull"], check=True)

KeyboardInterrupt: 

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q peft trl transformers accelerate bitsandbytes datasets \
    rouge-score sacrebleu bert-score python-dotenv pyyaml tqdm

In [ ]:
%cd language-engineer

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")
    f.write(f"HF_REPO_ID={HF_REPO_ID}\n")

## 2. (Optional) Mount Google Drive

Uncomment to persist checkpoints and datasets across Colab sessions.
Without this, all data is lost when the runtime disconnects.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# os.makedirs("/content/drive/MyDrive/lang-eng/runs", exist_ok=True)
# os.makedirs("/content/drive/MyDrive/lang-eng/datasets", exist_ok=True)
# if not os.path.exists("runs"):
#     os.symlink("/content/drive/MyDrive/lang-eng/runs", "runs")
# if not os.path.exists("datasets"):
#     os.symlink("/content/drive/MyDrive/lang-eng/datasets", "datasets")

## 3. Prepare Datasets

In [ ]:
!python scripts/prepare_datasets.py

In [ ]:
!python scripts/process_datasets.py

## 4. Train

In [ ]:
!python scripts/train.py --epochs {EPOCHS}

## 5. Evaluate

Compares base model (zero-shot) vs. fine-tuned adapter on the test set.
Results saved to `results/eval_results.json`.

In [ ]:
!python scripts/evaluate.py --n-samples {N_EVAL_SAMPLES}

## 6. Upload to HuggingFace Hub

In [ ]:
!python scripts/hub.py upload \
    --adapter runs/Qwen2.5-3B-Instruct_qlora/final \
    --repo {HF_REPO_ID}

## 7. (Optional) Download from HuggingFace Hub

Pull a previously uploaded adapter back to local disk (e.g. to resume eval on a new runtime).

In [ ]:
!python scripts/hub.py download \
    --repo {HF_REPO_ID} \
    --output runs/downloaded